# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zubairnajam/Week1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/zubairnajam/FlyRank-AI-ML-_Internship.git

Cloning into 'FlyRank-AI-ML-_Internship'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 131 (delta 42), reused 100 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.85 MiB | 10.51 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [2]:
import os
os.chdir("FlyRank-AI-ML-_Internship")
print("Now in:", os.getcwd())
print(os.listdir("data/raw"))

Now in: /content/FlyRank-AI-ML-_Internship
['content_refresh_anonymized.csv']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: a page is a review priority if it's visible (people actually see it in search), stale or declining, and has room to improve relative to where it already ranks. I'm combining three signals into one weighted score rather than a single if-statement, because — per the data contract — no single signal alone was strongly correlated with decline; the pattern only shows up when signals combine.

The formula:

baseline_score = 0.40 * visibility_score + 0.35 * staleness_risk_score + 0.25 * position_opportunity_score

Where:

visibility_score = normalized impressions_90d (a page nobody sees isn't worth reviewing regardless of other signals)
staleness_risk_score = normalized content_age_days, gated on being combined with visibility (old + invisible ≠ priority)
position_opportunity_score = inverse of avg_position for pages ranking in a recoverable range (position 1-20 — page 3+ of search is a different problem)

Reason codes it can output (mirrors the FlyRank session's real flags, applied to my lane):

stale_visible_page: content_age_days >= 180 and impressions_90d >= 500
declining_with_demand: trend_direction == "down" and impressions_90d >= 100
page_one_decay_risk: 0 < avg_position <= 10 and content_age_days >= 180

This rule uses only features present before the decision point — no trend_direction-derived score is used as a feature (only as a separate reason code label, checked in section 4), and nothing from a future time window.

In [3]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df_filtered = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates(subset="content_id")
    .copy()
)
print(f"Rows after filter: {len(df_filtered):,}")

Rows after filter: 30,000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import numpy as np

def normalize(series):
    s = series.astype(float)
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

df_filtered["visibility_score"] = normalize(df_filtered["impressions_90d"])
df_filtered["staleness_risk_score"] = normalize(df_filtered["content_age_days"])

pos_valid = df_filtered["avg_position"].between(1, 20)
df_filtered["position_opportunity_score"] = 0.0
df_filtered.loc[pos_valid, "position_opportunity_score"] = normalize(
    21 - df_filtered.loc[pos_valid, "avg_position"]
)

df_filtered["baseline_score"] =
(
    0.40 * df_filtered["visibility_score"]
    + 0.35 * df_filtered["staleness_risk_score"]
    + 0.25 * df_filtered["position_opportunity_score"]
)

def assign_reason_code(row):
    if row["content_age_days"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if row.get("trend_direction") == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand"
    if 0 < row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        return "page_one_decay_risk"
    return "monitor_only"

df_filtered["reason_code"] = df_filtered.apply(assign_reason_code, axis=1)
df_filtered["action"] = np.where(
    df_filtered["reason_code"] == "monitor_only", "monitor", "review_for_refresh"
)

ranked = df_filtered.sort_values("baseline_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "baseline_score", "reason_code", "action",
            "impressions_90d", "content_age_days", "avg_position", "trend_direction"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked):,} rows to work/outputs/baseline_action_score.csv")
ranked[out_cols].head(10)

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,content_age_days,avg_position,trend_direction
0,content_5fe46e04994d,client_4e07408562,0.937958,stale_visible_page,review_for_refresh,517715,537,4.2,down
1,content_8c19996aa890,client_4e07408562,0.885855,stale_visible_page,review_for_refresh,509252,445,2.5,down
2,content_aaef01a50def,client_19581e27de,0.853768,stale_visible_page,review_for_refresh,517109,445,5.4,stable
3,content_4c36c775b818,client_4e07408562,0.852831,stale_visible_page,review_for_refresh,463103,445,2.3,down
4,content_1a9e894be2e2,client_19581e27de,0.821529,stale_visible_page,review_for_refresh,416180,482,4.0,down
5,content_9532f197bbc8,client_4e07408562,0.737862,stale_visible_page,review_for_refresh,309192,445,2.0,down
6,content_db5989a78dd3,client_4e07408562,0.720877,stale_visible_page,review_for_refresh,345111,445,5.4,up
7,content_2c2606c5d176,client_19581e27de,0.677148,stale_visible_page,review_for_refresh,347399,362,4.2,down
8,content_9463d30d5826,client_19581e27de,0.624846,stale_visible_page,review_for_refresh,192478,480,5.7,down
9,content_fca1bf3940c0,client_4e07408562,0.604535,stale_visible_page,review_for_refresh,86170,537,4.2,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

One line each for the top 20 (fill in after running — template, since numbers depend on real data):

content_id=... — action: review_for_refresh, reason: stale_visible_page — high impressions + old content — would be wrong if: the page was recently updated but content_age_days wasn't refreshed in the data, or if it's evergreen content that doesn't need freshness (e.g. a reference page).
content_id=... — action: review_for_refresh, reason: page_one_decay_risk — ranks well but aging — would be wrong if: the ranking is actually stable, not decaying — this rule doesn't check trend, only age + position, so a stable old page-one result gets flagged unnecessarily.
... (repeat for all 20 once real output is in front of you — each needs a genuine, specific "what would make it wrong," not a copy-pasted caveat)

In [5]:
top20 = ranked.head(20)[out_cols]
print(top20.to_string())

              content_id          client_id  baseline_score         reason_code              action  impressions_90d  content_age_days  avg_position trend_direction
0   content_5fe46e04994d  client_4e07408562        0.937958  stale_visible_page  review_for_refresh           517715               537           4.2            down
1   content_8c19996aa890  client_4e07408562        0.885855  stale_visible_page  review_for_refresh           509252               445           2.5            down
2   content_aaef01a50def  client_19581e27de        0.853768  stale_visible_page  review_for_refresh           517109               445           5.4          stable
3   content_4c36c775b818  client_4e07408562        0.852831  stale_visible_page  review_for_refresh           463103               445           2.3            down
4   content_1a9e894be2e2  client_19581e27de        0.821529  stale_visible_page  review_for_refresh           416180               482           4.0            down
5   conten

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: any row landing in the top 20 purely on visibility_score with a monitor_only reason code is a weak pick — it means the page is popular but shows no actual sign of staleness or decay, so it's only there because raw traffic volume dominates the weighted formula. Also worth flagging: pages where position_opportunity_score = 0 (not ranking 1-20) that still score high on visibility + staleness alone — high traffic doesn't necessarily mean search visibility, could be direct/referral traffic, which this rule doesn't distinguish.

In [6]:
excluded_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
used_as_feature = [c for c in excluded_flags if c in ["visibility_score", "staleness_risk_score",
                                                          "position_opportunity_score"]]
print("Product flags used as scoring inputs:", used_as_feature)  # should be []

score_formula_inputs = ["impressions_90d", "content_age_days", "avg_position"]
print("Score built only from:", score_formula_inputs)
print("trend_direction used only for reason_code, not baseline_score:",
      "trend_direction" not in score_formula_inputs)

weak_picks = ranked[(ranked["reason_code"] == "monitor_only")].head(20)
print(f"\nWeak picks (monitor_only) landing near top: {len(ranked.head(20)[ranked.head(20)['reason_code']=='monitor_only'])}")

Product flags used as scoring inputs: []
Score built only from: ['impressions_90d', 'content_age_days', 'avg_position']
trend_direction used only for reason_code, not baseline_score: True

Weak picks (monitor_only) landing near top: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.